**Import Key Libraries**

In [12]:
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from urllib.parse import urlparse, parse_qs, urlencode, urlunparse
from datetime import datetime
import time

### Step1: Get all product Links

In [13]:
# ==========================================
# INPUTS
# ==========================================
search_box_text = "sports shoes for women"
website_link = "https://www.flipkart.com/"

# ==========================================
# SESSION START
# ==========================================
session_start_time = datetime.now().time()
print(f"\nSession Start Time: {session_start_time}")
print("=" * 80)

# ==========================================
# CHROME OPTIONS
# ==========================================
chrome_options = Options()

chrome_options.add_argument("--start-maximized")
chrome_options.add_argument("--disable-dev-shm-usage")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-gpu")

# Optional
chrome_options.add_argument("--disable-blink-features=AutomationControlled")

# ==========================================
# START DRIVER
# ==========================================
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 60)

try:

    # ==========================================
    # OPEN WEBSITE
    # ==========================================
    driver.get(website_link)

    print("Waiting for search box...")

    search_input = wait.until(
        EC.presence_of_element_located(
            (By.CSS_SELECTOR, '[autocomplete="off"]')
        )
    )

    print("Typing search query...")

    search_input.clear()
    search_input.send_keys(search_box_text)
    search_input.send_keys(Keys.RETURN)

    # ==========================================
    # WAIT FOR SEARCH RESULTS
    # ==========================================
    print("Waiting for search results...")

    wait.until(
        lambda d: d.execute_script(
            "return document.readyState"
        ) == "complete"
    )

    time.sleep(3)

    # ==========================================
    # GET FIRST PAGE URL
    # ==========================================
    first_page_link = driver.current_url

    print("\nFirst Search URL:")
    print(first_page_link)

    # ==========================================
    # CREATE PAGINATION LINKS
    # ==========================================
    all_pagination_links = []

    parsed_url = urlparse(first_page_link)

    for page_num in range(1, 26):

        query_params = parse_qs(parsed_url.query)

        query_params["page"] = [str(page_num)]

        new_query = urlencode(query_params, doseq=True)

        page_url = urlunparse(
            (
                parsed_url.scheme,
                parsed_url.netloc,
                parsed_url.path,
                parsed_url.params,
                new_query,
                parsed_url.fragment,
            )
        )

        all_pagination_links.append(page_url)

    print(f"\nPagination Links Generated: {len(all_pagination_links)}")

    # ==========================================
    # SCRAPE PRODUCT LINKS
    # ==========================================
    all_product_links = []

    print("\nCollecting Product Links...\n")

    for page_no, page_link in enumerate(all_pagination_links, start=1):

        try:

            # Check session alive
            driver.current_url

            print(f"Scraping Page {page_no}/25")

            driver.get(page_link)

            wait.until(
                lambda d: d.execute_script(
                    "return document.readyState"
                ) == "complete"
            )

            time.sleep(2)

            # Stable selector
            product_elements = wait.until(
                EC.presence_of_all_elements_located(
                    (
                        By.CSS_SELECTOR,
                        'a[href*="/p/"]'
                    )
                )
            )

            page_links = []

            for product in product_elements:

                href = product.get_attribute("href")

                if href and "/p/" in href:
                    page_links.append(href)

            page_links = list(set(page_links))

            print(
                f"Page {page_no}: "
                f"{len(page_links)} product links found"
            )

            all_product_links.extend(page_links)

        except Exception as e:

            print(
                f"Error on Page {page_no}"
            )

            print(str(e))

            continue

    # ==========================================
    # REMOVE DUPLICATES
    # ==========================================
    print("\nRemoving duplicates...")

    unique_links = list(set(all_product_links))

    df_product_links = pd.DataFrame(
        unique_links,
        columns=["product_links"]
    )

    print(
        f"\nTotal Unique Product Links: "
        f"{len(df_product_links)}"
    )

    # ==========================================
    # SAVE CSV
    # ==========================================
    output_file = "flipkart_product_links.csv"

    df_product_links.to_csv(
        output_file,
        index=False
    )

    print(
        f"\nCSV Saved Successfully:"
        f" {output_file}"
    )

except Exception as e:

    print("\nFatal Error:")
    print(str(e))

finally:

    try:
        driver.quit()
    except:
        pass

    session_end_time = datetime.now().time()

    print("\n" + "=" * 80)
    print(f"Session End Time: {session_end_time}")


Session Start Time: 17:15:06.953670
Waiting for search box...
Typing search query...
Waiting for search results...

First Search URL:
https://www.flipkart.com/search?q=sports%20shoes%20for%20women&otracker=search&otracker1=search&marketplace=FLIPKART&as-show=off&as=off

Pagination Links Generated: 25


Scraping Page 1/25
Page 1: 45 product links found
Scraping Page 2/25
Page 2: 45 product links found
Scraping Page 3/25
Page 3: 45 product links found
Scraping Page 4/25
Page 4: 45 product links found
Scraping Page 5/25
Page 5: 45 product links found
Scraping Page 6/25
Page 6: 45 product links found
Scraping Page 7/25
Page 7: 45 product links found
Scraping Page 8/25
Page 8: 45 product links found
Scraping Page 9/25
Page 9: 45 product links found
Scraping Page 10/25
Page 10: 45 product links found
Scraping Page 11/25
Page 11: 45 product links found
Scraping Page 12/25
Page 12: 45 product links found
Scraping Page 13/25
Page 13: 45 product links found
Scraping Page 14/25
Page 14: 45 produ

### Step2: Get Individual product information

In [14]:
import pandas as pd
import re
import time
from datetime import datetime

import undetected_chromedriver as uc

from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


# ==========================================
# DRIVER CREATION
# ==========================================

def create_driver():

    options = uc.ChromeOptions()

    options.add_argument("--start-maximized")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--no-sandbox")

    driver = uc.Chrome(
        version_main=149,      # Your Chrome version
        options=options,
        headless=False,
        use_subprocess=True
    )

    return driver


# ==========================================
# HELPER FUNCTIONS
# ==========================================

def browser_alive(driver):

    try:
        driver.current_url
        return True
    except:
        return False


def get_text(driver, xpath_list, timeout=5):

    for xpath in xpath_list:

        try:

            element = WebDriverWait(driver, timeout).until(
                EC.presence_of_element_located(
                    (By.XPATH, xpath)
                )
            )

            text = element.text.strip()

            if text:
                return text

        except:
            pass

    return ""


# ==========================================
# START SESSION
# ==========================================

session_start_time = datetime.now().time()

print(f"Session Start Time: {session_start_time}")

driver = create_driver()

# ==========================================
# LOAD LINKS
# ==========================================

df_links = pd.read_csv("flipkart_product_links.csv")

# testing
df_links = df_links.head(10)

all_product_links = df_links["product_links"].tolist()

print(f"Products to scrape: {len(all_product_links)}")

# ==========================================
# STORAGE
# ==========================================

products = []
unavailable_products = []

success_count = 0
failed_count = 0

# ==========================================
# MAIN LOOP
# ==========================================

for idx, url in enumerate(all_product_links, start=1):

    print(f"\n[{idx}/{len(all_product_links)}]")

    try:

        # -----------------------------------
        # RESTART DRIVER IF DEAD
        # -----------------------------------

        if not browser_alive(driver):

            print("Browser died. Restarting...")

            try:
                driver.quit()
            except:
                pass

            driver = create_driver()

        # -----------------------------------
        # OPEN PRODUCT PAGE
        # -----------------------------------

        driver.get(url)

        WebDriverWait(driver, 30).until(
            lambda d: d.execute_script(
                "return document.readyState"
            ) == "complete"
        )

        time.sleep(3)

        page_source = driver.page_source.lower()

        # -----------------------------------
        # BLOCK DETECTION
        # -----------------------------------

        if (
            "access denied" in page_source
            or "robot" in page_source
            or "captcha" in page_source
        ):

            print("Blocked by Flipkart")

            unavailable_products.append(url)

            continue

        # -----------------------------------
        # TITLE
        # -----------------------------------

        title = get_text(
            driver,
            [
                "//h1",
                "//span[contains(@class,'VU-ZEz')]"
            ]
        )

        title = re.sub(
            r"\s*\([^)]*\)",
            "",
            title
        )

        # -----------------------------------
        # BRAND
        # -----------------------------------

        brand = get_text(
            driver,
            [
                "//a[contains(@href,'brand')]",
                "//span[contains(@class,'mEh187')]"
            ]
        )

        if brand == "" and title:
            brand = title.split()[0]

        # -----------------------------------
        # PRICE
        # -----------------------------------

        price_text = get_text(
            driver,
            [
                "//div[contains(text(),'₹')]",
                "//span[contains(text(),'₹')]"
            ]
        )

        price_digits = re.findall(
            r"\d+",
            price_text
        )

        price = "".join(price_digits)

        # -----------------------------------
        # DISCOUNT
        # -----------------------------------

        discount_text = get_text(
            driver,
            [
                "//*[contains(text(),'% off')]"
            ]
        )

        discount_digits = re.findall(
            r"\d+",
            discount_text
        )

        if discount_digits:
            discount = int(discount_digits[0]) / 100
        else:
            discount = ""

        # -----------------------------------
        # RATING
        # -----------------------------------

        avg_rating = get_text(
            driver,
            [
                "//div[contains(@class,'XQDdHH')]"
            ]
        )

        total_ratings = ""

        try:

            ratings_text = get_text(
                driver,
                [
                    "//*[contains(text(),'Ratings')]"
                ]
            )

            nums = re.findall(
                r'[\d,]+',
                ratings_text
            )

            if nums:
                total_ratings = int(
                    nums[0].replace(",", "")
                )

        except:
            pass

        # -----------------------------------
        # SAVE
        # -----------------------------------

        products.append(
            [
                url,
                title,
                brand,
                price,
                discount,
                avg_rating,
                total_ratings
            ]
        )

        success_count += 1

        print(f"SUCCESS {success_count}")

        time.sleep(2)

    except Exception as e:

        failed_count += 1

        print(f"FAILED {failed_count}")
        print(str(e))

        unavailable_products.append(url)

        # Restart driver if session died

        if "invalid session id" in str(e).lower():

            try:
                driver.quit()
            except:
                pass

            driver = create_driver()

# ==========================================
# CREATE DATAFRAME
# ==========================================

df = pd.DataFrame(
    products,
    columns=[
        "product_link",
        "title",
        "brand",
        "price",
        "discount",
        "avg_rating",
        "total_ratings"
    ]
)

# ==========================================
# DUPLICATES
# ==========================================

duplicates = df[
    df.duplicated(
        subset=[
            "brand",
            "price",
            "discount",
            "avg_rating",
            "total_ratings"
        ]
    )
]

df = df.drop_duplicates(
    subset=[
        "brand",
        "price",
        "discount",
        "avg_rating",
        "total_ratings"
    ]
)

# ==========================================
# UNAVAILABLE
# ==========================================

df_unavailable = pd.DataFrame(
    unavailable_products,
    columns=["link"]
)

# ==========================================
# SAVE FILES
# ==========================================

df.to_csv(
    "flipkart_product_data.csv",
    index=False
)

duplicates.to_csv(
    "duplicate_products.csv",
    index=False
)

df_unavailable.to_csv(
    "unavailable_products.csv",
    index=False
)

# ==========================================
# SUMMARY
# ==========================================

print("\n" + "=" * 60)

print("Total URLs:", len(all_product_links))
print("Successful:", len(df))
print("Unavailable:", len(df_unavailable))
print("Duplicates:", len(duplicates))

# ==========================================
# CLOSE
# ==========================================

driver.quit()

session_end_time = datetime.now().time()

print(f"\nSession End Time: {session_end_time}")

Session Start Time: 17:16:48.663064
Products to scrape: 10

[1/10]
Blocked by Flipkart

[2/10]
Blocked by Flipkart

[3/10]
Blocked by Flipkart

[4/10]
Blocked by Flipkart

[5/10]
Blocked by Flipkart

[6/10]
Blocked by Flipkart

[7/10]
Blocked by Flipkart

[8/10]
Blocked by Flipkart

[9/10]
Blocked by Flipkart

[10/10]
Blocked by Flipkart

Total URLs: 10
Successful: 0
Unavailable: 10
Duplicates: 0

Session End Time: 17:17:38.830072
